In [1]:
import sys 

assert sys.version_info >= (3,10)

In [3]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [7]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [6]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("legend", fontsize=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [8]:
import deepxde as dde
import numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Setting up the backend


In [11]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")


Set the default float type to float64
Backend: pytorch


Exact solution for validation

In [12]:
def exact_solution(x):
    return (x + 1) ** 2

Domain geometry

In [13]:
geom = dde.geometry.Interval(-1, 1)

Define the Left and Right Boundary Conditions

In [19]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx - 2

def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def boundary_right(x, on_boudary):
    return on_boudary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)